In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_MARA_ODATA_CDS/ZCDS_MARA_ODATA"
username = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
password = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")

headers = {
    "Accept": "application/atom+xml",  # eksplisit minta XML
    "sap-client": "120"
}

params = {
    "$filter": "Ersda ge datetime'2019-01-01T00:00:00' and Ersda le datetime'2019-12-31T23:59:59'",
}

response = requests.get(
    url,
    auth=HTTPBasicAuth(username, password),
    headers=headers,
    params=params,
    verify=False,
    timeout=60
)
print(f"Status: {response.status_code}")
print(f"Content-Type: {response.headers.get('Content-Type')}")
print(f"Response length: {len(response.content)} bytes")
print("---RAW---")
print(response.text)

# Parse XML
try:    root = ET.fromstring(response.content)
except ET.ParseError as e:
    raise RuntimeError(f"Failed to parse XML: {e}")

# Parse XML
root = ET.fromstring(response.content)

ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "m": "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    "d": "http://schemas.microsoft.com/ado/2007/08/dataservices"
}

records = []

# Cek apakah root adalah <feed> (collection) atau <entry> (single)
root_tag = root.tag.split("}")[-1]

if root_tag == "feed":
    # Collection — loop semua entry
    for entry in root.findall("atom:entry", ns):
        props = entry.find(".//m:properties", ns)
        if props is not None:
            row = {child.tag.split("}")[-1]: child.text for child in props}
            records.append(row)

elif root_tag == "entry":
    # Single entity — langsung ambil properties dari root
    props = root.find(".//m:properties", ns)
    if props is not None:
        row = {child.tag.split("}")[-1]: child.text for child in props}
        records.append(row)

df = pd.DataFrame(records)
print(f"Total rows: {len(df):,}")
print(df.T)  # .T biar keliatan semua kolom (karena banyak field)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StringType

# ============================================================
# KNA1 Schema — SAP OData → Databricks Bronze
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType, 
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit

mara_schema = {
    "Matnr"               : StringType(),
    "Ersda"               : DateType(),          # DATS — Created On
    "Ernam"               : StringType(),
    "Laeda"               : DateType(),          # DATS — Last Changed Date
    "Aenam"               : StringType(),
    "Vpsta"               : StringType(),
    "Pstat"               : StringType(),
    "Lvorm"               : StringType(),
    "Mtart"               : StringType(),
    "Mbrsh"               : StringType(),
    "Matkl"               : StringType(),
    "Bismt"               : StringType(),
    "Meins"               : StringType(),        # UNIT
    "Bstme"               : StringType(),        # UNIT
    "Zeinr"               : StringType(),
    "Zeiar"               : StringType(),
    "Zeivr"               : StringType(),
    "Zeifo"               : StringType(),
    "Aeszn"               : StringType(),
    "Blatt"               : StringType(),
    "Blanz"               : StringType(),        # NUMC — keep string
    "Ferth"               : StringType(),
    "Formt"               : StringType(),
    "Groes"               : StringType(),
    "Wrkst"               : StringType(),
    "Normt"               : StringType(),
    "Labor"               : StringType(),
    "Ekwsl"               : StringType(),
    "Brgew"               : DecimalType(15, 3),  # QUAN — Gross Weight
    "Ntgew"               : DecimalType(15, 3),  # QUAN — Net Weight
    "Gewei"               : StringType(),        # UNIT
    "Volum"               : DecimalType(15, 3),  # QUAN — Volume
    "Voleh"               : StringType(),        # UNIT
    "Behvo"               : StringType(),
    "Raube"               : StringType(),
    "Tempb"               : StringType(),
    "Disst"               : StringType(),
    "Tragr"               : StringType(),
    "Stoff"               : StringType(),
    "Spart"               : StringType(),
    "Kunnr"               : StringType(),
    "Eannr"               : StringType(),
    "Wesch"               : DecimalType(15, 3),  # QUAN
    "Bwvor"               : StringType(),
    "Bwscl"               : StringType(),
    "Saiso"               : StringType(),
    "Etiar"               : StringType(),
    "Etifo"               : StringType(),
    "Entar"               : StringType(),
    "Ean11"               : StringType(),
    "Numtp"               : StringType(),
    "Laeng"               : DecimalType(15, 3),  # QUAN — Length
    "Breit"               : DecimalType(15, 3),  # QUAN — Width
    "Hoehe"               : DecimalType(15, 3),  # QUAN — Height
    "Meabm"               : StringType(),        # UNIT
    "Prdha"               : StringType(),
    "Aeklk"               : StringType(),
    "Cadkz"               : StringType(),
    "Qmpur"               : StringType(),
    "Ergew"               : DecimalType(15, 3),  # QUAN
    "Ergei"               : StringType(),        # UNIT
    "Ervol"               : DecimalType(15, 3),  # QUAN
    "Ervoe"               : StringType(),        # UNIT
    "Gewto"               : DecimalType(15, 3),  # DEC
    "Volto"               : DecimalType(15, 3),  # DEC
    "Vabme"               : StringType(),
    "Kzrev"               : StringType(),
    "Kzkfg"               : StringType(),
    "Xchpf"               : StringType(),
    "Vhart"               : StringType(),
    "Fuelg"               : DecimalType(15, 3),  # DEC
    "Stfak"               : IntegerType(),       # INT2
    "Magrv"               : StringType(),
    "Begru"               : StringType(),
    "Datab"               : DateType(),          # DATS — Valid From
    "Liqdt"               : DateType(),          # DATS — Discontinuation Date
    "Saisj"               : StringType(),
    "Plgtp"               : StringType(),
    "Mlgut"               : StringType(),
    "Extwg"               : StringType(),
    "Satnr"               : StringType(),
    "Attyp"               : StringType(),
    "Kzkup"               : StringType(),
    "Kznfm"               : StringType(),
    "Pmata"               : StringType(),
    "Mstae"               : StringType(),
    "Mstav"               : StringType(),
    "Mstde"               : DateType(),          # DATS
    "Mstdv"               : DateType(),          # DATS
    "Taklv"               : StringType(),
    "Rbnrm"               : StringType(),
    "Mhdrz"               : DecimalType(15, 3),  # DEC — Min Remaining Shelf Life
    "Mhdhb"               : DecimalType(15, 3),  # DEC — Total Shelf Life
    "Mhdlp"               : DecimalType(15, 3),  # DEC
    "Inhme"               : StringType(),        # UNIT
    "Inhal"               : DecimalType(15, 3),  # QUAN — Net Contents
    "Vpreh"               : DecimalType(15, 3),  # DEC
    "Etiag"               : StringType(),
    "Inhbr"               : DecimalType(15, 3),  # QUAN
    "Cmeth"               : StringType(),
    "Cuobf"               : StringType(),        # NUMC — keep string
    "Kzumw"               : StringType(),
    "Kosch"               : StringType(),
    "Sprof"               : StringType(),
    "Nrfhg"               : StringType(),
    "Mfrpn"               : StringType(),
    "Mfrnr"               : StringType(),
    "Bmatn"               : StringType(),
    "Mprof"               : StringType(),
    "Kzwsm"               : StringType(),
    "Saity"               : StringType(),
    "Profl"               : StringType(),
    "Ihivi"               : StringType(),
    "Iloos"               : StringType(),
    "Serlv"               : StringType(),
    "Kzgvh"               : StringType(),
    "Xgchp"               : StringType(),
    "Kzeff"               : StringType(),
    "Compl"               : StringType(),        # NUMC — keep string
    "Iprkz"               : StringType(),
    "Rdmhd"               : StringType(),
    "Przus"               : StringType(),
    "MtposMara"           : StringType(),
    "Bflme"               : StringType(),
    "Matfi"               : StringType(),
    "Cmrel"               : StringType(),
    "Bbtyp"               : StringType(),
    "SledBbd"             : StringType(),
    "GtinVariant"         : StringType(),
    "Gennr"               : StringType(),
    "Rmatp"               : StringType(),
    "GdsRelevant"         : StringType(),
    "Weora"               : StringType(),
    "HutypDflt"           : StringType(),
    "Pilferable"          : StringType(),
    "Whstc"               : StringType(),
    "Whmatgr"             : StringType(),
    "Hndlcode"            : StringType(),
    "Hazmat"              : StringType(),
    "Hutyp"               : StringType(),
    "TareVar"             : StringType(),
    "Maxc"                : DecimalType(15, 3),  # DEC
    "MaxcTol"             : DecimalType(15, 3),  # DEC
    "Maxl"                : DecimalType(15, 3),  # QUAN
    "Maxb"                : DecimalType(15, 3),  # QUAN
    "Maxh"                : DecimalType(15, 3),  # QUAN
    "MaxdimUom"           : StringType(),        # UNIT
    "Herkl"               : StringType(),
    "Mfrgr"               : StringType(),
    "Qqtime"              : DecimalType(15, 3),  # QUAN
    "Qqtimeuom"           : StringType(),        # UNIT
    "Qgrp"                : StringType(),
    "Serial"              : StringType(),
    "PsSmartform"         : StringType(),
    "Logunit"             : StringType(),        # UNIT
    "Cwqrel"              : StringType(),
    "Cwqproc"             : StringType(),
    "Cwqtolgr"            : StringType(),
    "Adprof"              : StringType(),
    "Ipmipproduct"        : StringType(),
    "AllowPmatIgno"       : StringType(),
    "Medium"              : StringType(),
    "Commodity"           : StringType(),
    "AnimalOrigin"        : StringType(),
    "TextileCompInd"      : StringType(),
    "LastChangedTime"     : StringType(),        # TIMS — keep string (ISO duration dari SAP)
    "MatnrExternal"       : StringType(),
    "LogisticalMatCategory": StringType(),
    "SalesMaterial"       : StringType(),
    "IdentificationTagType": StringType(),
    "SgtCsgr"             : StringType(),
    "SgtCovsa"            : StringType(),
    "SgtStat"             : StringType(),
    "SgtScope"            : StringType(),
    "SgtRel"              : StringType(),
    "Anp"                 : StringType(),        # NUMC — keep string
    "PsmCode"             : StringType(),
    "FshMgAt1"            : StringType(),
    "FshMgAt2"            : StringType(),
    "FshMgAt3"            : StringType(),
    "FshSealv"            : StringType(),
    "FshSeaim"            : StringType(),
    "FshScMid"            : StringType(),
    "DummyPrdInclEewPs"   : StringType(),
    "ScmMatidGuid16"      : StringType(),        # RAW — simpan as string
    "ScmMatidGuid22"      : StringType(),
    "ScmMaturityDur"      : DecimalType(15, 3),  # DEC
    "ScmShlfLfeReqMin"    : DecimalType(15, 3),  # DEC
    "ScmShlfLfeReqMax"    : DecimalType(15, 3),  # DEC
    "ScmPuom"             : StringType(),        # UNIT
    "RmatpPb"             : StringType(),
    "ProdShape"           : StringType(),
    "MoProfileId"         : StringType(),
    "OverhangTresh"       : DecimalType(15, 3),  # DEC
    "BridgeTresh"         : DecimalType(15, 3),  # DEC
    "BridgeMaxSlope"      : DecimalType(15, 3),  # DEC
    "HeightNonflat"       : DecimalType(15, 3),  # QUAN
    "HeightNonflatUom"    : StringType(),        # UNIT
    "xcwmxxcwmat"         : StringType(),
    "xcwmxvalum"          : StringType(),        # UNIT
    "xcwmxtolgr"          : StringType(),
    "xcwmxtara"           : StringType(),
    "xcwmxtarum"          : StringType(),        # UNIT
    "xbev1xluleinh"       : StringType(),        # NUMC — keep string
    "xbev1xluldegrp"      : StringType(),
    "xbev1xnestruccat"    : StringType(),
    "xdsdxslToltyp"       : StringType(),
    "xdsdxsvCntGrp"       : StringType(),
    "xdsdxvcGroup"        : StringType(),
    "xsapmpxkadu"         : DecimalType(15, 3),  # DEC
    "xsapmpxabmein"       : StringType(),        # UNIT
    "xsapmpxkadp"         : DecimalType(15, 3),  # DEC
    "xsapmpxbrad"         : DecimalType(15, 3),  # DEC
    "xsapmpxspbi"         : DecimalType(15, 3),  # DEC
    "xsapmpxtrad"         : DecimalType(15, 3),  # DEC
    "xsapmpxkedu"         : DecimalType(15, 3),  # DEC
    "xsapmpxsptr"         : DecimalType(15, 3),  # QUAN
    "xsapmpxfbdk"         : DecimalType(15, 3),  # DEC
    "xsapmpxfbhk"         : DecimalType(15, 3),  # DEC
    "xsapmpxrili"         : StringType(),
    "xsapmpxfbak"         : StringType(),
    "xsapmpxaho"          : StringType(),        # NUMC — keep string
    "xsapmpxmifrr"        : DecimalType(15, 3),  # DEC
    "xsttpecxsertype"     : IntegerType(),       # INT1
    "xsttpecxsyncact"     : StringType(),
    "xsttpecxsynctime"    : DecimalType(15, 3),  # DEC
    "xsttpecxsyncchg"     : StringType(),
    "xsttpecxcountryRef"  : StringType(),
    "xsttpecxprdcat"      : StringType(),
    "xvsoxrTiltInd"       : StringType(),
    "xvsoxrStackInd"      : StringType(),
    "xvsoxrBotInd"        : StringType(),
    "xvsoxrTopInd"        : StringType(),
    "xvsoxrStackNo"       : StringType(),        # NUMC — keep string
    "xvsoxrPalInd"        : StringType(),
    "xvsoxrPalOvrD"       : DecimalType(15, 3),  # QUAN
    "xvsoxrPalOvrW"       : DecimalType(15, 3),  # QUAN
    "xvsoxrPalBHt"        : DecimalType(15, 3),  # QUAN
    "xvsoxrPalMinH"       : DecimalType(15, 3),  # QUAN
    "xvsoxrTolBHt"        : DecimalType(15, 3),  # QUAN
    "xvsoxrNoPGvh"        : StringType(),        # NUMC — keep string
    "xvsoxrQuanUnit"      : StringType(),        # UNIT
    "xvsoxrKzgvhInd"      : StringType(),
    "Packcode"            : StringType(),
    "DgPackStatus"        : StringType(),
    "Mcond"               : StringType(),
    "Retdelc"             : StringType(),
    "LoglevReto"          : StringType(),
    "Nsnid"               : StringType(),
    "Ovlpn"               : StringType(),
    "AdspcSpc"            : StringType(),        # NUMC — keep string
    "Varid"               : StringType(),        # RAW — simpan as string
    "Msbookpartno"        : StringType(),
    "Dpcbt"               : StringType(),
    "Xgrdt"               : StringType(),
    "Imatn"               : StringType(),
    "Picnum"              : StringType(),
    "Bstat"               : StringType(),
    "ColorAtinn"          : StringType(),        # NUMC — keep string
    "Size1Atinn"          : StringType(),        # NUMC — keep string
    "Size2Atinn"          : StringType(),        # NUMC — keep string
    "Color"               : StringType(),
    "Size1"               : StringType(),
    "Size2"               : StringType(),
    "FreeChar"            : StringType(),
    "CareCode"            : StringType(),
    "BrandId"             : StringType(),
    "FiberCode1"          : StringType(),
    "FiberPart1"          : StringType(),        # NUMC — keep string
    "FiberCode2"          : StringType(),
    "FiberPart2"          : StringType(),        # NUMC — keep string
    "FiberCode3"          : StringType(),
    "FiberPart3"          : StringType(),        # NUMC — keep string
    "FiberCode4"          : StringType(),
    "FiberPart4"          : StringType(),        # NUMC — keep string
    "FiberCode5"          : StringType(),
    "FiberPart5"          : StringType(),        # NUMC — keep string
    "Fashgrd"             : StringType(),
}

spark = SparkSession.builder.getOrCreate()

# Step 1: pandas df → Spark df
spark_df = spark.createDataFrame(df)

# Step 2: Cast schema
for col_name, col_type in mara_schema.items():
    if col_name in spark_df.columns:
        if isinstance(col_type, DateType):
            # SAP OData kirim format ISO: "2019-08-23T00:00:00"
            spark_df = spark_df.withColumn(col_name, to_date(col(col_name), "yyyy-MM-dd'T'HH:mm:ss"))
        else:
            spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

# Step 3: Fix sisa kolom NullType
for field in spark_df.schema.fields:
    if isinstance(field.dataType, NullType):
        spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

# Step 4: Audit columns + write
spark_df = spark_df \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source", lit("ZCDS_MARA_ODATA"))

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("poc_enesis.bronze.ZCDS_MARA_ODATA")

print("✅ Bronze table berhasil dibuat!")
spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_MARA_ODATA").show()

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_MARA_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_MARA_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
